# RanPAC gốc · Algorithm 1 trên ImageNet-R

Dùng preset `7` của tác giả với **ViT-B/16 + SSF** ở Phase 1, rồi random projection + ReLU và ridge ở Phase 2. Preset dùng seed 1993, 10 task × 20 lớp, batch 48, 20 epoch cho Phase 1 và M = 10000. [Preset chính thức](https://github.com/McDonnell-Research-Lab/RanPAC/blob/cf4b301d18b0c27db030f4371b72b768005ae58a/args/imagenetr_publish.csv#L9)

1. Đưa **RanPAC-original-imagenetr.zip** vào thư mục gốc **My Drive**.
2. Tải **bản ImageNet-R đã chia train/test** từ liên kết mà repo tác giả cung cấp. Lưu file nén vào **My Drive** và sửa `dataset_input` ở cell bên dưới theo đúng tên. Notebook nhận ZIP, TAR/TAR.GZ hoặc một thư mục đã giải nén. Nếu dữ liệu tải về chỉ có 200 thư mục lớp mà thiếu `train/` và `test/`, hãy lấy lại bản có split từ các liên kết bên dưới.
3. Trong Colab, chọn **Runtime → Change runtime type → GPU**. Có thể thử T4 nếu được cấp. Nếu gặp CUDA out of memory ở batch 48, chọn GPU có nhiều VRAM hơn để giữ preset.
4. Chạy các bước 1–5 theo thứ tự. Cell W&B ở bước cuối dùng khi cần gửi kết quả lên project nhóm.

Dataset: [Google Drive](https://drive.google.com/file/d/1SG4TbiL8_DooekztyCVK8mPmfhMo8fkR/view?usp=sharing) · [OneDrive](https://entuedu-my.sharepoint.com/:u:/g/personal/n2207876b_e_ntu_edu_sg/EU4jyLL29CtBsZkB6y-JSbgBzWF5YHhBAUz1Qw8qM2954A?e=hlWpNW) · [README RanPAC](https://github.com/McDonnell-Research-Lab/RanPAC/blob/cf4b301d18b0c27db030f4371b72b768005ae58a/README.md)

## 1. Kết nối Drive và mở source

Sửa `dataset_input` theo tên file hoặc thư mục bạn đã lưu. Cho phép Colab truy cập Drive khi được hỏi.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import zipfile

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive")
source_zip = drive_root / "RanPAC-original-imagenetr.zip"
dataset_input = drive_root / "imagenet-r.zip"  # Sửa theo tên thật trên Drive.
repo_dir = Path("/content/RanPAC-original-imagenetr")
output_dir = drive_root / "RanPAC-original-imagenetr-results"

if not source_zip.is_file():
    raise FileNotFoundError(source_zip)

with zipfile.ZipFile(source_zip) as archive:
    archive.extractall("/content")
assert (repo_dir / "main.py").is_file()
output_dir.mkdir(exist_ok=True)

## 2. Cài môi trường Python 3.9 riêng

Dùng các phiên bản PyTorch/CUDA và timm của repo tác giả. Cài lại môi trường nếu Colab cấp runtime mới. [Hướng dẫn môi trường](https://github.com/McDonnell-Research-Lab/RanPAC#environment)

In [ ]:
%%bash
set -euo pipefail

cd /content
export MAMBA_ROOT_PREFIX=/content/ranpac-micromamba

if [[ ! -x /content/ranpac-env/bin/python ]]; then
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest \
    | tar -xj bin/micromamba
  /content/bin/micromamba create -y -p /content/ranpac-env \
    --override-channels -c conda-forge python=3.9 pip
fi

/content/ranpac-env/bin/python -m pip install \
  "torch==1.13.1+cu117" "torchvision==0.14.1+cu117" "torchaudio==0.13.1" \
  --extra-index-url https://download.pytorch.org/whl/cu117

/content/ranpac-env/bin/python -m pip install \
  "numpy==1.24.4" "pandas==1.5.2" "tqdm==4.65.0" "timm==0.6.12"

## 3. Kiểm tra GPU và import SSF

Cell cần in tên GPU và dòng xác nhận SSF đã được đăng ký trong timm.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
python_bin = "/content/ranpac-env/bin/python"
subprocess.run(
    [python_bin, "-u", "-c", """
import sys
import timm, torch, torchvision, pandas
from petl import vision_transformer_ssf

assert torch.cuda.is_available(), "PyTorch không thấy GPU"
x = torch.ones((16, 16), device="cuda")
assert (x @ x)[0, 0].item() == 16
assert timm.is_model("vit_base_patch16_224_ssf")
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| SSF: OK")
"""],
    cwd=repo_dir,
    check=True,
);

## 4. Chuẩn bị split ImageNet-R

Notebook đưa dữ liệu vào ổ đĩa của runtime để huấn luyện, rồi dùng `ImageFolder` kiểm tra `train/` và `test/` có cùng 200 lớp. Chờ cell in số ảnh của hai split trước khi chạy. Nếu đổi nguồn dataset, hãy dùng một runtime mới để tránh dùng lại dữ liệu đã giải nén.

In [ ]:
from tempfile import TemporaryDirectory
import shutil
import tarfile

dataset_dir = repo_dir / "data" / "imagenet-r"
dataset_dir.parent.mkdir(exist_ok=True)

def validate_dataset(root):
    subprocess.run(
        [python_bin, "-u", "-c", """
from pathlib import Path
import sys
from torchvision.datasets import ImageFolder

root = Path(sys.argv[1])
train = ImageFolder(root / "train")
test = ImageFolder(root / "test")
if len(train.classes) != 200 or train.class_to_idx != test.class_to_idx:
    raise ValueError("Train và test phải có cùng 200 lớp ImageNet-R.")
for split_name, split in (("train", train), ("test", test)):
    split[0]
    print(f"{split_name}: {len(split):,} ảnh, {len(split.classes)} lớp")
""", str(root)],
        check=True,
    )

if dataset_dir.is_dir():
    print("Đang dùng lại dataset đã chuẩn bị trong runtime.")
    validate_dataset(dataset_dir)
else:
    if not dataset_input.exists():
        raise FileNotFoundError(f"Sửa dataset_input theo tên trên Drive: {dataset_input}")
    with TemporaryDirectory(dir="/content") as temp:
        stage = Path(temp) / "extracted"
        stage.mkdir()
        if dataset_input.is_dir():
            shutil.copytree(dataset_input, stage / "dataset")
        else:
            if zipfile.is_zipfile(dataset_input):
                with zipfile.ZipFile(dataset_input) as archive:
                    archive.extractall(stage)
            elif tarfile.is_tarfile(dataset_input):
                with tarfile.open(dataset_input) as archive:
                    archive.extractall(stage, filter="data")
            else:
                raise ValueError("Dataset cần ở dạng ZIP, TAR/TAR.GZ hoặc thư mục.")

        split_roots = [
            train_dir.parent for train_dir in stage.rglob("train")
            if train_dir.is_dir() and (train_dir.parent / "test").is_dir()
        ]
        if len(split_roots) != 1:
            raise ValueError("Cần đúng một thư mục chứa cả train/ và test/. Kiểm tra bản split.")
        validate_dataset(split_roots[0])
        split_roots[0].rename(dataset_dir)

## 5. Chạy Algorithm 1

Cell gọi `python main.py -i 7 -d imagenetr` với preset nguyên bản. Lần đầu repo tải trọng số pretrained. Theo dõi `console.log` ở đường dẫn in ra đầu cell. SSF cần GPU và bước tìm λ ở Phase 2 chạy trên CPU.

Mỗi lượt lưu log, đường accuracy và dự đoán vào một thư mục riêng trong `MyDrive/RanPAC-original-imagenetr-results/`. Nếu runtime ngắt, các file đã ghi vẫn nằm trên Drive. Repo ghi CSV theo task và lượt chạy mới sẽ bắt đầu lại từ đầu. Run hoàn tất cần có 10 dòng `ave_acc`. Dòng cuối là A[10].

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import csv

def read_curve(run_dir):
    path = run_dir / "results" / "imagenetr_publish_7.csv"
    with path.open(newline="") as file:
        return path, list(csv.DictReader(file))

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
run_dir = output_dir / f"preset-7-{stamp}-{uuid4().hex[:12]}"
run_dir.mkdir()
for name in ("results", "logs"):
    (run_dir / name).mkdir()

print("Đang ghi log và kết quả vào:", run_dir, flush=True)
with TemporaryDirectory(dir="/content") as temp:
    work_dir = Path(temp)
    for name in ("args", "data"):
        (work_dir / name).symlink_to(repo_dir / name, target_is_directory=True)
    for name in ("results", "logs"):
        (work_dir / name).symlink_to(run_dir / name, target_is_directory=True)

    with (run_dir / "console.log").open("w") as log:
        process = subprocess.run(
            [python_bin, "-u", str(repo_dir / "main.py"), "-i", "7", "-d", "imagenetr"],
            cwd=work_dir,
            stdout=log,
            stderr=subprocess.STDOUT,
        )

(run_dir / "exit_code.txt").write_text(str(process.returncode))
process.check_returncode()
csv_path, rows = read_curve(run_dir)
assert len(rows) == 10, f"CSV có {len(rows)} task. Xem console.log."
print(f"A[10] = {float(rows[-1]['ave_acc']):.2f}%")
print(csv_path)

## 6. Đưa các lượt hoàn tất lên W&B

Thêm `WANDB_API_KEY` trong **Colab → Secrets** và bật **Notebook access**. Đổi `entity, project` nếu cần. Mỗi lượt đủ 10 task được ghi thành một W&B run, với đường `A_t`, summary `A_10` và CSV đi kèm. Có thể chạy lại cell sau mỗi đợt để upload các lượt mới. Nếu mở runtime mới chỉ để upload, chạy lại bước 1 để mount Drive và khai báo đường dẫn trước.

In [ ]:
%pip install -q wandb

import csv
import wandb
from google.colab import userdata

entity, project = "pnp123", "ML"
wandb.login(key=userdata.get("WANDB_API_KEY"))

for folder in sorted(output_dir.glob("preset-7-*")):
    marker = folder / f"uploaded-{entity}-{project}.txt"
    exit_file = folder / "exit_code.txt"
    csv_path = folder / "results" / "imagenetr_publish_7.csv"
    if marker.exists() or not exit_file.is_file() or not csv_path.is_file():
        continue
    if exit_file.read_text().strip() != "0":
        continue
    with csv_path.open(newline="") as file:
        rows = list(csv.DictReader(file))
    if len(rows) != 10:
        print(f"Bỏ qua {folder.name}: CSV có {len(rows)} task.")
        continue

    with wandb.init(
        entity=entity, project=project, group="ranpac-original-imagenetr",
        name=folder.name, id=folder.name.rsplit("-", 1)[-1], resume="allow",
        config={"dataset": "imagenetr", "preset": 7, "seed": 1993,
                "petl": "ssf", "M": 10000, "batch_size": 48, "epochs": 20,
                "source_commit": "cf4b301d18b0c27db030f4371b72b768005ae58a"},
    ) as run:
        run.define_metric("task")
        run.define_metric("A_t", step_metric="task")
        for task, row in enumerate(rows, start=1):
            run.log({"task": task, "A_t": float(row["ave_acc"])}, step=task)
        run.summary["A_10"] = float(rows[-1]["ave_acc"])
        artifact = wandb.Artifact(f"ranpac-{run.id}", type="results")
        artifact.add_file(str(csv_path))
        run.log_artifact(artifact).wait()
        run_url = run.url

    marker.write_text(run_url)
    print(folder.name, run_url)